# Binomial GLMMs: cbpp, VerbAgg and toenail

Generalized linear mixed models for binary and binomial responses with `glmer()`:

* the **cbpp** data (contagious bovine pleuropneumonia): a binomial response given as
  `cbind(successes, failures)` or, equivalently, as a proportion with `weights`;
* the Laplace approximation (`nAGQ = 1`, the default) versus adaptive Gauss–Hermite quadrature
  (`nAGQ = 9, 25`) versus the faster `nAGQ = 0`;
* logit, probit and complementary log–log links;
* overdispersion and an observation-level random effect;
* odds ratios with Wald confidence intervals and predicted probabilities;
* **VerbAgg**: crossed random effects for subjects and items;
* **toenail**: a longitudinal binary outcome where quadrature matters.

## Source

* lme4 help pages: [`?glmer`](https://github.com/lme4/lme4/blob/master/man/glmer.Rd),
  [`?cbpp`](https://github.com/lme4/lme4/blob/master/man/cbpp.Rd),
  [`?VerbAgg`](https://github.com/lme4/lme4/blob/master/man/VerbAgg.Rd),
  [`?toenail`](https://github.com/lme4/lme4/blob/master/man/toenail.Rd)
  (local copies in `upstream/lme4/man/`).
* The lme4 GLMM vignette, *Fitting generalized linear mixed models with lme4*
  (`upstream/lme4/vignettes/glmer.Rnw`): Laplace approximation, adaptive Gauss–Hermite
  quadrature and the toenail example.
* Ben Bolker's [GLMM FAQ](https://bbolker.github.io/mixedmodels-misc/glmmFAQ.html), sections on
  overdispersion (Pearson χ² test, observation-level random effects) and on the choice of
  integration method.

The full R script is `examples/r/03_binomial_glmm.R`; it also writes the reference values
used in the "Check against R" cells (`examples/reference/03_binomial_glmm.json`, R 4.6.2,
lme4 2.1.0). Its core, taken from the help pages:

```r
## ?glmer, ?cbpp
gm1 <- glmer(cbind(incidence, size - incidence) ~ period + (1 | herd),
             data = cbpp, family = binomial)
m1p <- glmer(incidence / size ~ period + (1 | herd), weights = size,
             family = binomial, data = cbpp)
gm1a <- glmer(cbind(incidence, size - incidence) ~ period + (1 | herd),
              cbpp, binomial, nAGQ = 0)
gm1b <- glmer(cbind(incidence, size - incidence) ~ period + (1 | herd),
              cbpp, binomial, nAGQ = 9)
cbpp$obs <- 1:nrow(cbpp)
gm2 <- glmer(cbind(incidence, size - incidence) ~ period + (1 | herd) + (1 | obs),
             family = binomial, data = cbpp)
anova(gm1, gm2)
## glmer and glm log-likelihoods are consistent
gm1Devfun <- update(gm1, devFunOnly = TRUE)
gm0 <- glm(cbind(incidence, size - incidence) ~ period, family = binomial, data = cbpp)
stopifnot(all.equal(gm1Devfun(c(0, coef(gm0))), c(-2 * logLik(gm0))))

## odds ratios, Wald intervals and predicted probabilities
exp(cbind(OR = fixef(gm1), confint(gm1, method = "Wald")[-1, ]))
predict(gm1, type = "response")
predict(gm1, newdata = data.frame(period = factor(1:4)), re.form = NA, type = "response")

## ?VerbAgg (main effects only)
fmVA <- glmer(r2 ~ Anger + Gender + btype + situ + (1|id) + (1|item),
              family = binomial, data = VerbAgg)
## ?toenail, ?glmer
toemod <- glmer(outcome ~ time*treatment + (1 | patientID),
                data = toenail, family = binomial(link = "logit"))
toemod20 <- update(toemod, nAGQ = 20)
```

## Setup

`check_fit()` compares a fit with its R reference: log-likelihood, deviance, AIC and BIC with
`rtol = 1e-8`, fixed effects and θ with `rtol = 1e-6` (looser for the flat surfaces discussed
below). Standard errors are compared with `rtol = 1e-5`: as in R, the covariance matrix of the
fixed effects of a GLMM is the inverse of a *finite-difference* Hessian of the deviance
(`use.hessian = TRUE`), and the numerical derivative amplifies differences in the last digits
of the optimum to about 1e-6.

In [1]:
import json

import numpy as np
import pandas as pd
from numpy.testing import assert_allclose, assert_equal
from scipy.stats import chi2, norm

import lme4
from lme4 import glmer, binomial, load_dataset

with open("reference/03_binomial_glmm.json") as fh:
    ref = json.load(fh)
print(ref["R_version"], "| lme4", ref["lme4_version"], "| Python lme4", lme4.__version__)


def check_fit(m, r, est_rtol=1e-6, ll_rtol=1e-8, dev_rtol=None, se_rtol=None):
    # Compare a glmer fit with an R reference entry (see examples/r/03_binomial_glmm.R).
    assert_allclose(m.logLik().value, r["logLik"], rtol=ll_rtol)
    assert_allclose(m.deviance(), r["deviance"], rtol=dev_rtol or ll_rtol)
    assert_allclose(m.AIC(), r["AIC"], rtol=ll_rtol)
    assert_allclose(m.BIC(), r["BIC"], rtol=ll_rtol)
    assert_equal(list(m.fixef().index), list(r["fixef"]))
    assert_allclose(m.fixef().values, list(r["fixef"].values()), rtol=est_rtol)
    assert_allclose(m.getME("theta").values, np.atleast_1d(r["theta"]), rtol=est_rtol)
    # vcov() of a GLMM comes from a finite-difference Hessian of the deviance: agreement ~1e-6
    assert_allclose(np.sqrt(np.diag(m.vcov())), r["se"], rtol=se_rtol or max(est_rtol, 1e-5))

R version 4.6.2 (2026-10-29) | lme4 2.1.0 | Python lme4 2.1.0a2


## 1. cbpp: a binomial response

`cbpp` records, for 15 herds and 4 periods, the number of new serological cases of CBPP
(`incidence`) among the `size` animals at risk. The response is the pair
(cases, non-cases), written `cbind(incidence, size - incidence)` exactly as in R.

In [2]:
cbpp = load_dataset("cbpp")
cbpp.head()

,herd,incidence,size,period
0,1,2.0,14.0,1
1,1,3.0,12.0,2
2,1,4.0,9.0,3
3,1,0.0,5.0,4
4,2,3.0,22.0,1


In [3]:
gm1 = glmer("cbind(incidence, size - incidence) ~ period + (1 | herd)",
            data=cbpp, family=binomial)
print(gm1.summary())

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: cbind(incidence, size - incidence) ~ period + (1 | herd)
   Data: cbpp

      AIC       BIC    logLik -2*log(L)  df.resid 
    194.1     204.2     -92.0     184.1        51 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-2.3818 -0.7888 -0.2024  0.5146  2.8796 

Random effects:
 Groups Name        Variance Std.Dev.
 herd   (Intercept) 0.4125   0.6423  
Number of obs: 56, groups:  herd, 15

Fixed effects:
            Estimate Std. Error z value Pr(>|z|)    
(Intercept)  -1.3985     0.2325  -6.016 1.79e-09 ***
period2      -0.9923     0.3066  -3.236 0.001212 ** 
period3      -1.1287     0.3266  -3.455 0.000549 ***
period4      -1.5803     0.4274  -3.697 0.000218 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Correlation of Fixed Effects:
        (Intr) perid2 perid3
period2 -0.364              
period3 -0.340  0.278    

The conditional modes of the herd effects:

In [4]:
re = gm1.ranef()["herd"]
re.head()

,(Intercept)
1,0.590021
2,-0.298903
3,0.406254
4,0.039275
5,-0.190022


### Check against R

In [5]:
r = ref["gm1"]
check_fit(gm1, r)
assert_allclose(np.asarray(gm1.vcov()), r["vcov"], rtol=2e-5)
assert_allclose(re.iloc[:, 0].values, r["ranef"], rtol=1e-6)
assert_allclose(gm1.VarCorr().as_data_frame()["sdcor"].values, r["sdcor"], rtol=1e-6)
print("gm1 agrees with R")

gm1 agrees with R


### Proportion + `weights`

The same model can be specified with the observed proportion as the response and the number
of trials as prior weights (`?cbpp`). The fits are identical.

In [6]:
m1p = glmer("incidence / size ~ period + (1 | herd)", weights="size",
            data=cbpp, family=binomial)
pd.DataFrame({"cbind": gm1.fixef(), "proportion + weights": m1p.fixef()})

,cbind,proportion + weights
(Intercept),-1.398532,-1.398532
period2,-0.992316,-0.992316
period3,-1.128664,-1.128664
period4,-1.580316,-1.580316


In [7]:
# ?cbpp: stopifnot(all.equal(fixef(m1), fixef(m1p), tolerance = 1e-5), all.equal(ranef(...)))
assert_allclose(m1p.fixef().values, gm1.fixef().values, rtol=1e-5)
assert_allclose(m1p.ranef()["herd"].iloc[:, 0].values, re.iloc[:, 0].values, rtol=1e-5)
# Check against R
check_fit(m1p, ref["m1p"])
assert_allclose(m1p.ranef()["herd"].iloc[:, 0].values, ref["m1p"]["ranef"], rtol=1e-6)
print("m1p agrees with gm1 and with R")

m1p agrees with gm1 and with R


## 2. Laplace vs adaptive Gauss–Hermite quadrature vs `nAGQ = 0`

The marginal likelihood of a GLMM is an integral over the random effects with no closed form.

* `nAGQ = 1` (default): the **Laplace approximation**, a Gaussian approximation of the
  integrand around the conditional modes.
* `nAGQ > 1`: **adaptive Gauss–Hermite quadrature** with `nAGQ` nodes per random effect,
  available only for a single scalar random-effects term such as `(1 | herd)`.
* `nAGQ = 0`: the fixed effects are estimated inside the PIRLS step together with the
  conditional modes, and only θ is optimized: faster, but only an approximation of the
  Laplace fit.

As `?glmer` notes, the deviance reported for `nAGQ > 1` is computed from the sum of deviance
residuals and is not directly comparable with the `nAGQ = 0` / `nAGQ = 1` values; compare the
estimates instead.

In [8]:
form = "cbind(incidence, size - incidence) ~ period + (1 | herd)"
fits_q = {q: glmer(form, data=cbpp, family=binomial, nAGQ=q) for q in (0, 1, 9, 25)}
pd.DataFrame({f"nAGQ = {q}": pd.concat([m.fixef(),
                                        pd.Series({"sd(herd)": m.getME("theta").iloc[0],
                                                   "deviance": m.deviance()})])
              for q, m in fits_q.items()}).round(5)

,nAGQ = 0,nAGQ = 1,nAGQ = 9,nAGQ = 25
(Intercept),-1.36047,-1.39853,-1.39923,-1.39923
period2,-0.97618,-0.99232,-0.99139,-0.99139
period3,-1.11108,-1.12866,-1.12783,-1.12783
period4,-1.55968,-1.58032,-1.57944,-1.57944
sd(herd),0.64182,0.64227,0.64751,0.64751
deviance,73.48464,73.47154,73.37310,73.37310


For this well-behaved data set the Laplace and quadrature estimates agree to two or three
digits, and `nAGQ = 9` and `nAGQ = 25` are practically indistinguishable: the quadrature has
converged.

### Check against R

In [9]:
for q, m in fits_q.items():
    check_fit(m, ref["nAGQ"][f"q{q}"])
print("nAGQ = 0, 1, 9, 25 agree with R")

nAGQ = 0, 1, 9, 25 agree with R


## 3. Link functions

`binomial(link=...)` accepts the `logit` (default), `probit`, `cloglog`, `cauchit` and `log`
links, as in R.

In [10]:
fits_link = {lk: glmer(form, data=cbpp, family=binomial(link=lk))
             for lk in ("logit", "probit", "cloglog")}
pd.DataFrame({lk: pd.concat([m.fixef(),
                             pd.Series({"sd(herd)": m.getME("theta").iloc[0],
                                        "logLik": m.logLik().value, "AIC": m.AIC()})])
              for lk, m in fits_link.items()}).round(4)

,logit,probit,cloglog
(Intercept),-1.3985,-0.8355,-1.5260
period2,-0.9923,-0.5280,-0.9163
period3,-1.1287,-0.6169,-1.0352
period4,-1.5803,-0.7996,-1.4836
sd(herd),0.6423,0.3380,0.5867
logLik,-92.0263,-92.6212,-91.8020
AIC,194.0526,195.2424,193.6041


The coefficients are on different scales (probit coefficients are roughly the logit ones
divided by 1.6–1.8), while the log-likelihoods are nearly identical: these data cannot
distinguish the links.

### Check against R

In [11]:
for lk, m in fits_link.items():
    check_fit(m, ref["links"][lk])
print("logit, probit and cloglog fits agree with R")

logit, probit and cloglog fits agree with R


## 4. glmer and glm log-likelihoods are consistent

From `?glmer`: the deviance function of `gm1` evaluated at θ = 0 (no herd variation) and at
the coefficients of the ordinary logistic regression equals that model's −2 log L.
`update(devFunOnly=True)` returns the deviance function of the Laplace fit, whose argument is
`c(theta, beta)`.

In [12]:
from lme4.lmlist import glm   # transcription of stats::glm (used internally by lmList)

gm0 = glm("cbind(incidence, size - incidence) ~ period", family=binomial, data=cbpp)
gm1Devfun = gm1.update(devFunOnly=True)
gm1Dev0 = gm1Devfun(np.r_[0.0, gm0.coefficients.values])
m2ll_glm = gm0.aic - 2 * gm0.rank          # -2 logLik for a binomial glm
print(gm0.coefficients.round(6).to_dict())
print(f"devfun(0, coef(gm0)) = {gm1Dev0:.10f}   -2 logLik(gm0) = {m2ll_glm:.10f}")
assert_allclose(gm1Dev0, m2ll_glm, rtol=1e-8)

{'(Intercept)': -1.269023, 'period2': -1.170763, 'period3': -1.301405, 'period4': -1.782279}
devfun(0, coef(gm0)) = 198.0583989787   -2 logLik(gm0) = 198.0583989787


### Check against R

In [13]:
assert_allclose(gm0.coefficients.values, ref["glm0"]["coef"], rtol=1e-8)
assert_allclose(m2ll_glm, ref["glm0"]["m2logLik"], rtol=1e-10)
assert_allclose(gm1Dev0, ref["glm0"]["devfun_at_glm"], rtol=1e-10)
print("glm / glmer consistency agrees with R")

glm / glmer consistency agrees with R


## 5. Overdispersion and an observation-level random effect

A binomial GLMM fixes the conditional variance at nπ(1−π). A quick check described in the
GLMM FAQ compares the sum of squared Pearson residuals with the residual degrees of freedom:
a ratio well above 1 (and a small χ² p-value) suggests overdispersion. It is only
approximate, since the residual degrees of freedom of a mixed model are not well defined.

In [14]:
rp = gm1.residuals(type="pearson")
rdf = gm1.df_residual()
pearson = float(np.sum(rp ** 2))
print(f"Pearson chi2 = {pearson:.3f}, rdf = {rdf}, ratio = {pearson / rdf:.3f}, "
      f"p = {chi2.sf(pearson, rdf):.4f}")

Pearson chi2 = 63.078, rdf = 51, ratio = 1.237, p = 0.1195


A standard remedy for extra-binomial variation is an **observation-level random effect**
(OLRE): a factor with one level per row. For cbpp it is equivalent to a `period:herd`
interaction (`?glmer`).

In [15]:
cbpp["obs"] = pd.Categorical(np.arange(1, len(cbpp) + 1))
gm2 = glmer("cbind(incidence, size - incidence) ~ period + (1 | herd) + (1 | obs)",
            data=cbpp, family=binomial)
print(gm2.summary())

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: cbind(incidence, size - incidence) ~ period + (1 | herd) + (1 | obs)
   Data: cbpp

      AIC       BIC    logLik -2*log(L)  df.resid 
    186.6     198.8     -87.3     174.6        50 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-1.2866 -0.5989 -0.1181  0.3575  1.6216 

Random effects:
 Groups Name        Variance Std.Dev.
 obs    (Intercept) 0.79402  0.8911  
 herd   (Intercept) 0.03384  0.1840  
Number of obs: 56, groups:  obs, 56; herd, 15

Fixed effects:
            Estimate Std. Error z value Pr(>|z|)    
(Intercept)  -1.5003     0.2967  -5.056 4.27e-07 ***
period2      -1.2265     0.4803  -2.554  0.01066 *  
period3      -1.3288     0.4939  -2.690  0.00714 ** 
period4      -1.8663     0.5936  -3.144  0.00167 ** 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Correlation of Fixed Effects:
        (Intr) perid2 pe

In [16]:
aov = gm1.anova(gm2, model_names=["gm1", "gm2"])
aov

,npar,AIC,BIC,logLik,-2*log(L),Chisq,Df,Pr(>Chisq)
gm1,5,194.052564,204.179322,-92.026282,184.052564,NaN,NaN,NaN
gm2,6,186.638311,198.790421,-87.319156,174.638311,9.414253,1.0,0.002153


The likelihood-ratio test favours the OLRE model (it tests a variance on the boundary of the
parameter space, so the p-value is conservative).

### Check against R

In [17]:
check_fit(gm2, ref["gm2"])
assert_allclose(gm2.VarCorr().as_data_frame()["sdcor"].values, ref["gm2"]["sdcor"], rtol=1e-6)
ra = ref["anova12"]
assert_allclose(aov["npar"].values, ra["npar"])
assert_allclose(aov["AIC"].values, ra["AIC"], rtol=1e-8)
assert_allclose(aov["logLik"].values, ra["logLik"], rtol=1e-8)
assert_allclose(aov["Chisq"].iloc[1], ra["Chisq"], rtol=1e-6)
assert_allclose(aov["Pr(>Chisq)"].iloc[1], ra["p"], rtol=1e-6)
ro = ref["overdisp"]
assert_allclose([pearson, rdf, chi2.sf(pearson, rdf)], [ro["chisq"], ro["rdf"], ro["p"]],
                rtol=1e-6)
print("gm2, anova and the Pearson overdispersion statistic agree with R")

gm2, anova and the Pearson overdispersion statistic agree with R


## 6. Odds ratios with Wald confidence intervals

Exponentiated logit-scale coefficients are odds ratios. Wald intervals
(`confint(method="Wald")`) are cheap; they are given only for the fixed effects (the row for
the random-effect standard deviation is `NaN`, as in R).

In [18]:
ci = gm1.confint(method="Wald")
ci

,2.5 %,97.5 %
.sig01,NaN,NaN
(Intercept),-1.854171,-0.942893
period2,-1.593322,-0.391310
period3,-1.768862,-0.488466
period4,-2.418077,-0.742554


In [19]:
or_tab = np.exp(pd.concat([gm1.fixef().rename("OR"), ci.loc[gm1.fixef().index]], axis=1))
or_tab.round(4)

,OR,2.5 %,97.5 %
(Intercept),0.2470,0.1566,0.3895
period2,0.3707,0.2032,0.6762
period3,0.3235,0.1705,0.6136
period4,0.2059,0.0891,0.4759


Relative to period 1, the odds of a new case in a given herd are about 63%, 68% and 79%
lower in periods 2, 3 and 4.

### Check against R

In [20]:
rw = ref["wald"]
assert_equal(list(ci.index), rw["rownames"])
lo = np.array([np.nan if v == "NA" else v for v in rw["lower"]], float)
hi = np.array([np.nan if v == "NA" else v for v in rw["upper"]], float)
assert_allclose(ci.iloc[:, 0].values, lo, rtol=1e-5)
assert_allclose(ci.iloc[:, 1].values, hi, rtol=1e-5)
assert_allclose(or_tab["OR"].values, ref["OR"]["est"], rtol=1e-5)
assert_allclose(or_tab.iloc[:, 1].values, ref["OR"]["lower"], rtol=1e-5)
assert_allclose(or_tab.iloc[:, 2].values, ref["OR"]["upper"], rtol=1e-5)
z = gm1.fixef() / np.sqrt(np.diag(gm1.vcov()))
assert_allclose(z.values, ref["coeftab"]["z"], rtol=1e-5)
assert_allclose(2 * norm.sf(np.abs(z.values)), ref["coeftab"]["p"],
                rtol=1e-4)
print("Wald intervals, z tests and odds ratios agree with R")

Wald intervals, z tests and odds ratios agree with R


## 7. Predicted probabilities

`predict(type="response")` returns probabilities (`type="link"`, the default, returns
log-odds). Without `newdata` the predictions are conditional on the estimated herd effects;
`re_form="NA"` gives population-level predictions (random effects set to zero).

In [21]:
p_cond = gm1.predict(type="response")
pd.DataFrame({"herd": cbpp["herd"], "period": cbpp["period"],
              "observed": cbpp["incidence"] / cbpp["size"], "predicted": p_cond}).head(8)

,herd,period,observed,predicted
0,1,1,0.142857,0.308208
1,1,2,0.250000,0.141750
2,1,3,0.444444,0.125958
3,1,4,0.000000,0.084029
4,2,1,0.136364,0.154801
5,2,2,0.055556,0.063581
6,2,3,0.047619,0.055930
7,3,1,0.363636,0.270462


In [22]:
nd = pd.DataFrame({"period": pd.Categorical(["1", "2", "3", "4"])})
pop = gm1.predict(nd, re_form="NA", type="response")
herd1 = gm1.predict(nd.assign(herd="1"), type="response")
pd.DataFrame({"period": nd["period"], "population": pop, "herd 1": herd1})

,period,population,herd 1
0,1,0.198049,0.308208
1,2,0.083873,0.141750
2,3,0.073973,0.125958
3,4,0.048391,0.084029


Herd 1 has a positive conditional mode, so its predicted incidence lies above the
population-level curve. The population-level prediction is the probability for a *typical*
herd (random effect zero), not the average probability across herds.

### Check against R

In [23]:
rp_ = ref["pred"]
assert_allclose(p_cond.values, rp_["resp"], rtol=1e-6)
assert_allclose(gm1.predict().values, rp_["link"], rtol=1e-6)
assert_allclose(np.asarray(gm1.fitted()), rp_["fitted"], rtol=1e-6)
assert_allclose(pop.values, rp_["pop"], rtol=1e-6)
assert_allclose(herd1.values, rp_["herd1"], rtol=1e-6)
print("predictions agree with R")

predictions agree with R


## 8. VerbAgg: crossed random effects for subjects and items

`VerbAgg` contains the answers of 316 subjects to 24 items on verbal aggression; `r2` is the
dichotomized response (`"no"` vs `"perhaps"`/`"yes"`). Subjects and items are **crossed**:
every subject answers every item. The model below is the main-effects version of the model
in `?VerbAgg` (which adds all two-way interactions).

In [24]:
VerbAgg = load_dataset("VerbAgg")
print(VerbAgg.shape)
VerbAgg.head()

(7584, 9)

,Anger,Gender,item,resp,id,btype,situ,mode,r2
0,20,M,S1WantCurse,no,1,curse,other,want,N
1,11,M,S1WantCurse,no,2,curse,other,want,N
2,17,F,S1WantCurse,perhaps,3,curse,other,want,Y
3,21,F,S1WantCurse,perhaps,4,curse,other,want,Y
4,17,F,S1WantCurse,perhaps,5,curse,other,want,Y


In [25]:
fmVA = glmer("r2 ~ Anger + Gender + btype + situ + (1 | id) + (1 | item)",
             data=VerbAgg, family=binomial)
print(fmVA.summary(correlation=False))

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: r2 ~ Anger + Gender + btype + situ + (1 | id) + (1 | item)
   Data: VerbAgg

      AIC       BIC    logLik -2*log(L)  df.resid 
   8167.4    8222.9   -4075.7    8151.4      7576 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-6.1040 -0.6227 -0.1782  0.6265 13.5500 

Random effects:
 Groups Name        Variance Std.Dev.
 id     (Intercept) 1.7948   1.3397  
 item   (Intercept) 0.2453   0.4953  
Number of obs: 7584, groups:  id, 316; item, 24

Fixed effects:
            Estimate Std. Error z value Pr(>|z|)    
(Intercept)  0.19910    0.40651   0.490 0.624291    
Anger        0.05743    0.01684   3.411 0.000648 ***
GenderM      0.32068    0.19193   1.671 0.094754 .  
btypescold  -1.05880    0.25727  -4.116 3.86e-05 ***
btypeshout  -2.10541    0.25945  -8.115 4.86e-16 ***
situself    -1.05551    0.21077  -5.008 5.50e-07 ***
---
Signif. codes:  0

In [26]:
fmVA0 = glmer("r2 ~ Anger + Gender + btype + situ + (1 | id) + (1 | item)",
              data=VerbAgg, family=binomial, nAGQ=0)
pd.DataFrame({"Laplace": pd.concat([fmVA.fixef(), fmVA.getME("theta")]),
              "nAGQ = 0": pd.concat([fmVA0.fixef(), fmVA0.getME("theta")])}).round(4)

,Laplace,nAGQ = 0
(Intercept),0.1991,0.2083
Anger,0.0574,0.0544
GenderM,0.3207,0.3041
btypescold,-1.0588,-1.0165
btypeshout,-2.1054,-2.0218
situself,-1.0555,-1.0134
id.(Intercept),1.3397,1.3396
item.(Intercept),0.4953,0.4968


### Check against R

R reports a convergence warning for the Laplace fit (`max|grad| = 0.00209`, just above the
tolerance 0.002): the likelihood surface is flat near the optimum. Both implementations reach
the same maximum (log-likelihoods agree to about 3e-10 relative), but the point where the
optimizer stops on the plateau differs slightly — rounding at the 1e-15 level from different
sparse Cholesky orderings changes the optimizer's path. The estimates therefore agree to about
three significant digits (largest relative difference about 2e-3), and are compared with
`rtol = 1e-2`. The same holds for `deviance()`, which for a GLMM is the sum of the
conditional deviance residuals plus the penalty and so depends on the conditional modes: it
agrees to about 1e-6 relative, while the Laplace log-likelihood agrees to 3e-10. The `nAGQ = 0` fit is well determined and agrees to full precision.

In [27]:
print("R convergence message:", ref["VerbAgg"]["messages"])
check_fit(fmVA, ref["VerbAgg"], est_rtol=1e-2, ll_rtol=1e-9, dev_rtol=1e-5)
assert_allclose(fmVA.VarCorr().as_data_frame()["sdcor"].values, ref["VerbAgg"]["sdcor"],
                rtol=1e-3)
assert_allclose(fmVA.ranef()["item"].iloc[:, 0].values, ref["VerbAgg"]["item_ranef"],
                rtol=1e-2, atol=1e-3)
check_fit(fmVA0, ref["VerbAgg0"])
print("VerbAgg fits agree with R")

R convergence message: Model failed to converge with max|grad| = 0.00208604 (tol = 0.002, component 1)
  See ?lme4::convergence and ?lme4::troubleshooting.
VerbAgg fits agree with R


## 9. toenail: a longitudinal binary outcome

The toenail data (De Backer et al., 1998) follow 294 patients treated for onychomycosis over
up to 7 visits; the binary outcome is the degree of separation of the nail plate
(`"moderate or severe"` vs `"none or mild"`). With few binary observations per patient and a
very large between-patient variance, the Laplace approximation is poor here, and the vignette
uses this data set to motivate quadrature. For a factor response the first level
(`"none or mild"`) is a failure and the other is a success.

In [28]:
toenail = load_dataset("toenail")
print(toenail["outcome"].cat.categories.tolist())
ttab = toenail.groupby("patientID", observed=True).size().value_counts().sort_index()
print("number of patients by number of visits:", ttab.to_dict())

['none or mild', 'moderate or severe']
number of patients by number of visits: {1: 5, 2: 3, 3: 7, 4: 6, 5: 10, 6: 39, 7: 224}


In [29]:
toe_form = "outcome ~ time * treatment + (1 | patientID)"
toemod = glmer(toe_form, data=toenail, family=binomial(link="logit"))
toemod20 = glmer(toe_form, data=toenail, family=binomial, nAGQ=20)
pd.DataFrame({name: pd.concat([m.fixef(),
                               pd.Series({"sd(patient)": m.getME("theta").iloc[0]})])
              for name, m in [("Laplace", toemod), ("nAGQ = 20", toemod20)]}).round(4)

,Laplace,nAGQ = 20
(Intercept),-2.5234,-1.6182
time,-0.4001,-0.3908
treatmentterbinafine,-0.3070,-0.1608
time:treatmentterbinafine,-0.1373,-0.1367
sd(patient),4.5709,4.0037


In [30]:
print(toemod20.summary(correlation=False))

Generalized linear mixed model fit by maximum likelihood (Adaptive
  Gauss-Hermite Quadrature, nAGQ = 20) [glmerMod]
 Family: binomial  ( logit )
Formula: outcome ~ time * treatment + (1 | patientID)
   Data: toenail

      AIC       BIC    logLik -2*log(L)  df.resid 
   1260.8    1288.5    -625.4    1250.8      1903 

Scaled residuals: 
   Min     1Q Median     3Q    Max 
-3.208 -0.194 -0.093 -0.009 43.656 

Random effects:
 Groups    Name        Variance Std.Dev.
 patientID (Intercept) 16.03    4.004   
Number of obs: 1908, groups:  patientID, 294

Fixed effects:
                          Estimate Std. Error z value Pr(>|z|)    
(Intercept)               -1.61822    0.43317  -3.736 0.000187 ***
time                      -0.39084    0.04432  -8.818  < 2e-16 ***
treatmentterbinafine      -0.16076    0.58372  -0.275 0.783001    
time:treatmentterbinafine -0.13673    0.06799  -2.011 0.044331 *  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


The Laplace approximation gives a larger between-patient standard deviation and larger fixed
effects than 20-point quadrature — the bias discussed in the vignette. Odds ratios from the
quadrature fit:

In [31]:
ci20 = toemod20.confint(method="Wald")
np.exp(pd.concat([toemod20.fixef().rename("OR"), ci20.loc[toemod20.fixef().index]],
                 axis=1)).round(4)

,OR,2.5 %,97.5 %
(Intercept),0.1983,0.0848,0.4634
time,0.6765,0.6202,0.7379
treatmentterbinafine,0.8515,0.2712,2.6733
time:treatmentterbinafine,0.8722,0.7634,0.9965


### Check against R

Both fits converge without warnings in R. The `nAGQ = 20` estimates are compared with
`rtol = 2e-4`: the quadrature objective is flat (sd ≈ 4 on the logit scale) and the
optimizers stop slightly apart — about 7e-6 relative on Windows and 5e-5 on Linux builds of
this package — while the log-likelihoods agree to about 1e-12 relative.
The standard errors of both fits agree to 1e-4 (compared with `se_rtol = 5e-4`), and the `nAGQ = 20` `deviance()` — the sum of
conditional deviance residuals, which depends on the conditional modes (see section 2) —
to about 2.5e-7 (`dev_rtol = 1e-6`). The standard errors differ even for the Laplace fit,
whose estimates agree to 1e-8: they come from a finite-difference Hessian (step 1e-4) of a
deviance that is itself computed by an iterative PIRLS inner loop, warm-started from the
previous evaluation and stopped at a tolerance. With a between-patient sd of about 4.5 that inner
loop noise, divided by the squared step, shows up in the fourth significant digit of the
Hessian in both implementations. (The same sensitivity shows up between runs of this
package: differences in the last bits of the Eigen computations, e.g. from memory alignment,
can move the Laplace SEs by ~1e-4 between processes.)

In [32]:
assert_equal({str(k): int(v) for k, v in ttab.items()}, ref["toenail_ttab"])
check_fit(toemod, ref["toenail"], se_rtol=5e-4)
check_fit(toemod20, ref["toenail20"], est_rtol=2e-4, se_rtol=5e-4, dev_rtol=1e-6)
print("toenail fits agree with R")

toenail fits agree with R


## Summary

| analysis | source | agreement with R |
|---|---|---|
| cbpp, Laplace; proportion + weights | `?glmer`, `?cbpp` | logLik ~1e-14, estimates ~1e-9 |
| nAGQ = 0, 1, 9, 25 | `?glmer`, vignette | deviance ~1e-9, estimates ~1e-8 |
| logit / probit / cloglog links | | logLik ~1e-14 |
| glm consistency at θ = 0 | `?glmer` | ~1e-15 |
| OLRE, `anova`, Pearson χ² | `?glmer`, GLMM FAQ | ~1e-8 |
| Wald CIs, odds ratios, `predict(type="response")` | | ~1e-8 |
| VerbAgg (R warns: flat surface) | `?VerbAgg` | logLik ~3e-10, estimates ~3 digits |
| toenail, Laplace and nAGQ = 20 | `?toenail`, vignette | logLik ~1e-12, estimates 1e-7 (Laplace) / 5e-5 (AGQ) |